# E0: Head–Representation Compatibility Controls

## Objective
Explore the completed, closed E0.1/E0.2 experiment using saved aggregate summaries. Five folds × three encoder seeds × three VQC initializations produced 45 head refits, plus 15 LR-angle fits. Original B and LR-z remain historical references. No fitting, model construction, checkpoints, or cohort access occurs in this notebook.

The prespecified E0.1 primary effects are training and validation log loss versus historical B; E0.2 primary effects are paired validation ROC-AUC and log loss of LR-angle minus LR-z. The diagonal h=s subset is displayed separately without adding fits or treating it as a pure initialization contrast.

Observed means: VQC refit−B training loss −0.000220 nats and validation loss −0.000382 nats; LR-angle−LR-z validation AUC −0.000068 and loss +0.002029 nats. Effects vary across encoders and folds. These descriptive differences are not equivalence tests or independent-cohort confidence estimates.

See the [self-contained report](../reports/phase2/e0_v1/report.md), [approved protocol](../reports/phase2/e0_protocol_v1.md), and [methodological review](../reports/phase2/e0_methodology_review_v1.md).

## Load existing artifacts
Run from the repository root or `notebooks/`. Set `QARL_E0_ANALYSIS_DIR` to another completed summary directory if needed. Private predictions, models, and detailed provenance remain in Git-ignored `runs/phase2/e0_v1/`. The notebook needs only aggregate CSVs and the existing analysis/notebook dependencies.

In [ ]:
import os
from pathlib import Path
import sys
import matplotlib.pyplot as plt
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "src/phase2").is_dir():
    raise RuntimeError("Run from the repository root or notebooks directory.")
sys.path.insert(0, str(root))
from src.phase2.e0_views import (load_tables, plot_e01, plot_diagonal, plot_e02,
    plot_initializations, plot_metrics, plot_curves, plot_generalization, plot_cost)
analysis = Path(os.environ.get("QARL_E0_ANALYSIS_DIR", str(root / "reports/phase2/e0_v1")))
tables = load_tables(analysis)
plt.rcParams.update({"font.size": 10, "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
fold_to_display = 0  # Display-only choice; no checkpoint or model selection.


## Diagnostics and visualizations
### E0.1 primary effects: fit and generalization
Every point is a paired comparison against the historical head on the same final B embedding. Negative differences mean lower loss. Squares identify the prespecified diagonal. The 45 heads are dependent outcomes sharing encoders and validation patients, not 45 independent replications.

In [ ]:
display(tables["primary_summary"].round(7))
plot_e01(tables["e01_paired"])
plt.show()

### Prespecified diagonal h=s versus historical B
The same initial head seed does not recreate joint training: the historical encoder coevolved with its head, whereas a refit sees final z from its first step. These 15 comparisons are already included in the 45 fits. Loss decreases in 14 training cases and seven validation cases; effects are heterogeneous.

In [ ]:
plot_diagonal(tables["e01_diagonal"])
plt.show()
display(tables["e01_diagonal"][["fold", "encoder_seed", "train_log_loss_reference_B",
    "train_log_loss", "train_log_loss_minus_B", "validation_log_loss_reference_B",
    "validation_log_loss", "validation_log_loss_minus_B"]].round(7))

### All three initializations on every encoder
All three outcomes are retained, with identical minibatch streams within each encoder. The mean within-encoder validation-loss SD is 0.000171 nats, much smaller than the observed variation between encoder seeds. This does not prove that every initialization or another budget would behave similarly, or explain how the encoder was learned.

In [ ]:
plot_initializations(tables["e01_paired"])
plt.show()

### E0.2 primary effects: LR-angle versus LR-z
LR-z is the unchanged historical probe on z. LR-angle uses the same estimator settings and training patients on π·tanh(z/2). No extra scaling, C search, calibration, or threshold selection occurs. Same C does not imply the same effective regularization under this nonlinear map. Ranking and probability-loss results need not agree.

In [ ]:
plot_e02(tables["e02_paired"])
plt.show()
display(tables["e02_paired"][["fold", "encoder_seed", "validation_roc_auc_minus_LR_z",
    "validation_log_loss_minus_LR_z"]].round(7))

### All planned validation metrics
ER+=1, ER−=0; AP ER− uses 1−p. Decisions use p≥0.5. Lower log loss and Brier are better. VQC heads are averaged within each encoder before plotting comparable encoder-level results. Black marks show fold means. Spread is descriptive, without confidence intervals or a best-head selection.

In [ ]:
plot_metrics(tables["metrics"])
plt.show()
display(tables["summary"].query("split == 'validation' and metric in ['roc_auc', 'ap_er_negative', 'balanced_accuracy', 'log_loss', 'brier']").round(6))

### Same-checkpoint training and validation
Final training losses are evaluated at the same checkpoint as validation. Mean VQC refit training/validation losses are 0.048583/0.286290 nats, compared with 0.048803/0.286672 for historical B. LR-angle fits training better than LR-z (0.092192 versus 0.110383) without improving mean validation loss (0.283954 versus 0.281925). Lower training loss alone does not establish improved generalization.

In [ ]:
plot_generalization(tables["head_results"])
plt.show()

### Fixed-budget optimization histories
All 50 epochs are retained; the final epoch is the only evaluation checkpoint. Dashed training curves are online minibatch losses and are distinct from final-checkpoint training losses above. The display parameter selects an existing fold, without fitting or choosing a favorable epoch. Independent panel y scales show trajectories.

In [ ]:
plot_curves(tables["curves"], fold=fold_to_display)
plt.show()

### Conditional computational cost
These are measured current-process head costs on saved embeddings, excluding the sunk cost of learning B. The VQC and L-BFGS budgets are different. Timing includes validation/diagnostics as indicated, is host-load dependent, and is not a hardware or quantum-advantage benchmark. Historical costs were measured separately and are not pooled with these timings.

In [ ]:
plot_cost(tables["costs"])
plt.show()
display(tables["costs"].groupby("model")[["fit_update_seconds", "epoch_validation_seconds",
    "final_prediction_seconds", "iterations"]].agg(["mean", "min", "max"]).round(6))

## Interpretation
The tested fresh heads largely recover the historical VQC's fit on fixed z. The small training-loss reductions and mixed validation effects do not support a substantial head-restart remedy under this budget; they do not prove global optimality or equivalence. Classical discrimination is similar before and after the map, while training loss improves and validation loss is mixed. This weakens a blanket explanation based solely on poor linear readability of angular features, without establishing map invariance or the source of representation quality.

For C0, the findings motivate distinguishing representation-learning mechanisms from readout fitting, retaining both classical feature-space controls and fixed-head-budget baselines. They do not select a geometric objective or prove that a regularizer would help. The approved methodological review distinguishes the existing single-example BCE from pairwise metric-learning objectives.

## Limitations
The encoders were already learned jointly with VQCs and are not neutral representations. Head-initialization interventions with fixed batches are controlled conditional comparisons; refits versus historical B also change the training trajectory. Three starts, one optimizer, one budget, and these overlapping development folds do not identify global head capacity, encoder causality, or population effects. No E0.3, C0, C1, extra epochs, alternate optimizer, map, architecture, or validation-selected search was executed. Locked-test data remain untouched.